In [1]:
#Importar librerías
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal
from rdflib.namespace import RDF, OWL, XSD

In [2]:
#Leer CSV
data = pd.read_csv("turismo.csv")

print("Número de registros:", len(data))

data.head()

Número de registros: 190


,nombre,codigo_iso,wikidata_qid,latitud,longitud,anio,viajeros,pernoctaciones
0,Andalucía,ES-AN,Q5783,37.405,-5.9875,2016,21182420,65878086
1,Andalucía,ES-AN,Q5783,37.405,-5.9875,2017,22154747,68351904
2,Andalucía,ES-AN,Q5783,37.405,-5.9875,2018,22660180,69094618
3,Andalucía,ES-AN,Q5783,37.405,-5.9875,2019,24043131,71546175
4,Andalucía,ES-AN,Q5783,37.405,-5.9875,2020,8634567,23884824


In [3]:

g = Graph() #Crear grafo donde meter las tripeltas

SCHEMA = Namespace("https://schema.org/")
EX = Namespace("https://example.org/turismo/")
WD = Namespace("http://www.wikidata.org/entity/")

g.bind("schema", SCHEMA) #clases y propiedades proporcionadas por Schema.org
g.bind("ex", EX) #identificadores de los recursos que nosotros vamos a crear
g.bind("wd", WD) #identificadores de recursos que ya existen en Wikidata
g.bind("owl", OWL)

In [4]:
#Transformar las comunidades en tripletas
for _, row in data.iterrows():

    ccaa = URIRef(EX + "ccaa/" + row["codigo_iso"])

    # Clase y propiedades de la comunidad autónoma
    g.add((ccaa, RDF.type, SCHEMA.AdministrativeArea)) #Andalucía es una región administrativa
    g.add((ccaa, SCHEMA.name, Literal(row["nombre"]))) #Añadimos el nombre a la ca ("Andalucía")
    g.add((ccaa, SCHEMA.identifier, Literal(row["codigo_iso"]))) #Añadimos su identificador iso ("ES-AN")
    
    """
    <https://example.org/turismo/ccaa/ES-AN>
    a schema:AdministrativeArea ;
    schema:name "Andalucía" ;
    schema:identifier "ES-AN" 
    """

    # Enlace con Wikidata
    if pd.notna(row["wikidata_qid"]): #Si identificador != null
        wikidata = URIRef(WD + row["wikidata_qid"])
        g.add((ccaa, OWL.sameAs, wikidata)) #Relación con wikidata https://www.wikidata.org/wiki/Q5783

    # Coordenadas geográficas
    if pd.notna(row["latitud"]) and pd.notna(row["longitud"]):
    
        geo = URIRef(EX + "geo/" + row["codigo_iso"])
        g.add((ccaa, SCHEMA.geo, geo))
        g.add((geo, RDF.type, SCHEMA.GeoCoordinates))
        g.add((geo, SCHEMA.latitude, Literal(row["latitud"])))
        g.add((geo, SCHEMA.longitude, Literal(row["longitud"])))

In [5]:
#Transformar datos turísticos en tripletas
for _, row in data.iterrows():
    ccaa = URIRef(EX + "ccaa/" + row["codigo_iso"])
    anio = int(row["anio"])

    for medida in ["viajeros", "pernoctaciones"]:

        if pd.isna(row[medida]):
            continue

        observacion = URIRef(EX + f"observacion/{row['codigo_iso']}/{anio}/{medida}")

        g.add((observacion, RDF.type, SCHEMA.Observation)) #El recurso en una observación estadística

        g.add((observacion, SCHEMA.observationAbout, ccaa)) #La observación se refiere a Andalucía

        g.add((observacion, SCHEMA.observationDate, Literal(f"{anio}-01-01", datatype=XSD.date))) #El preriodo empieza el 1 del 1 de 2025
        g.add((observacion, SCHEMA.observationPeriod, Literal("P1Y"))) #El periodo comprende un año

        g.add((observacion, SCHEMA.variableMeasured,Literal(medida))) #Medimos viajeros
        g.add((observacion, SCHEMA.value,Literal(int(row[medida])))) #El número de viajeros es [medida]

In [6]:
#Generar RDF
g.serialize(destination="turismo.ttl", format="turtle")


print("Tripletas generadas:", len(g))

Tripletas generadas: 2432


In [7]:
#Check de datos generados
rdf = Graph()

rdf.parse("turismo.ttl", format="turtle")

print("Número de tripletas:", len(rdf))

assert len(rdf) == len(g)

Número de tripletas: 2432
